In [2]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt


# Step 1: Load data

df = pd.read_csv("allframe_update_addEpige.txt", sep="\t", low_memory=False)


# Step 2: Filter valid guides

df = df.dropna(subset=["Guide_sequence"])
df = df[df["Guide_sequence"].str.match("^[ACGTN]{18,24}$", na=False)]


# Step 3: Convert success column

df["success_clean"] = df["success"].astype(str).str.strip().str.lower()

mapping = {
    "success": 1,
    "true": 1,
    "yes": 1,
    "1": 1,
    "fail": 0,
    "false": 0,
    "no": 0,
    "0": 0
}

df["success_num"] = df["success_clean"].map(mapping)
df = df.dropna(subset=["success_num"])
df["success_num"] = df["success_num"].astype(float)


# Step 4: Compute off-target score

df["Mismach"] = pd.to_numeric(df["Mismach"], errors="coerce").fillna(0)
df["Bulge"] = pd.to_numeric(df["Bulge"], errors="coerce").fillna(0)

df["off"] = (df["Mismach"] + df["Bulge"]) / (df["Mismach"] + df["Bulge"] + 1)

# Step 5: Normalize for scoring

df["norm_eff"] = df["success_num"] / df["success_num"].max()
df["norm_off"] = 1 - df["off"]

# Widgets

max_off_widget = widgets.FloatSlider(
    value=0.30, min=0.10, max=1.00, step=0.05,
    description='Max Off-target:',
    continuous_update=True,
    readout=True,
    readout_format='.2f'
)

min_eff_widget = widgets.FloatSlider(
    value=50.0, min=0.0, max=100.0, step=1.0,
    description='Min Efficiency:',
    continuous_update=True,
    readout=True,
    readout_format='.0f'
)

weight_eff_widget = widgets.FloatSlider(
    value=0.40, min=0.0, max=1.0, step=0.05,
    description='Weight Efficiency:',
    continuous_update=True,
    readout=True,
    readout_format='.2f'
)

output = widgets.Output()

def evaluate_guides(change=None):
    with output:
        clear_output()

        MAX_OFF = max_off_widget.value
        MIN_EFF = min_eff_widget.value / 100.0
        WEIGHT_EFF = weight_eff_widget.value
        WEIGHT_OFF = 1 - WEIGHT_EFF

        df["score"] = round(df["norm_eff"] * WEIGHT_EFF + df["norm_off"] * WEIGHT_OFF, 3)

        df["status"] = df.apply(
            lambda r: "Rejected: High off-target" if r["off"] > MAX_OFF else
                      ("Rejected: Low efficiency" if r["success_num"] < MIN_EFF else "Accepted"),
            axis=1
        )

        accepted = df[df["status"] == "Accepted"].sort_values(by="score", ascending=False)
        rejected = df[df["status"] != "Accepted"]

        print(f"Total guides evaluated: {len(df)}")
        print(f"Accepted guides: {len(accepted)}")
        print(f"Rejected guides: {len(rejected)}\n")

        if not accepted.empty:
            display(accepted[["Guide_sequence","success_num","off","score","status"]].head(20))

        if not rejected.empty:
            print("Rejected guides:")
            display(rejected[["Guide_sequence","success_num","off","score","status"]].head(20))

        # Plot
        plt.figure(figsize=(9,6))
        plt.scatter(df["success_num"]*100, df["off"],
                    c=df["status"].apply(lambda x: 'green' if x=="Accepted" else 'red'),
                    s=80)

        plt.xlabel("Efficiency (%)")
        plt.ylabel("Off-target risk")
        plt.title("CRISPR Guide Evaluation")
        plt.axhline(MAX_OFF, linestyle="--")
        plt.axvline(min_eff_widget.value, linestyle="--")
        plt.show()

max_off_widget.observe(evaluate_guides, names="value")
min_eff_widget.observe(evaluate_guides, names="value")
weight_eff_widget.observe(evaluate_guides, names="value")

display(max_off_widget, min_eff_widget, weight_eff_widget, output)

evaluate_guides()




FloatSlider(value=0.3, description='Max Off-target:', max=1.0, min=0.1, step=0.05)

FloatSlider(value=50.0, description='Min Efficiency:', readout_format='.0f', step=1.0)

FloatSlider(value=0.4, description='Weight Efficiency:', max=1.0, step=0.05)

Output()